# 觀光旅館營運月報 — 資料讀取與清洗

觀光署的觀光旅館月報橫跨 2017-2025 年共 108 個月份，檔案格式混雜（htm、xlsx、ods、pdf），
且不同年份的欄位名稱、表頭位置、資料結構都不一致。

本 notebook 負責：
1. 批次讀取資料夾內所有月報檔案，自動辨識檔案格式
2. 統一清洗：定位表頭與結尾列、移除小計列、正規化欄位名稱
3. 合併為單一 DataFrame 並輸出為 `觀光旅館_all.xlsx`
4. 標記異常月份（201901、201906、202003）

In [19]:
import os
import re
import pandas as pd
import numpy as np
import tabula  # pip install tabula-py（需要 Java 環境）
import unicodedata
import warnings
warnings.filterwarnings('ignore')


In [ ]:

FOLDER = r"C:\Users\User\Desktop\taiwan_hotel_analysis\觀光旅館"  # 改成你的實際路徑

df_dict= {}
# 找檔案 (ex: 201701月報表.htm)
for filename in os.listdir(FOLDER): 
    filepath = os.path.join(FOLDER, filename) # ex:C:\Users\User\Desktop\taiwan_hotel_analysis\觀光旅館\201701月報表.htm

    # 如果資料夾中還有其他資料夾，跳過資料夾
    if os.path.isdir(filepath):
        continue

    # 從檔名開頭抓 YYYYMM（6 碼數字）
    match = re.match(r"(\d{6})", filename)
    if not match:
        print(f"[跳過] 無法辨識年月: {filename}")
        continue

    ym = match.group(1)  # ex "201701"

    # 取副檔名（小寫）
    ext = os.path.splitext(filename)[1].lower() # ex: htm

    # 依照檔案類型取出df
    try:
        if ext == ".html" or ext == ".htm":
            # read_html 回傳 list[DataFrame]，取最大的那張表
            df = pd.read_html(filepath, encoding="utf-8")[0]

        elif ext == ".xlsx" or ext == ".xls":
            df = pd.read_excel(filepath)

        elif ext == ".ods":
            df = pd.read_excel(filepath, engine="odf")
            
        elif ext == ".pdf":
            df = tabula.read_pdf(filepath, pages="all", lattice=True, encoding="cp950")[0]
        else:
            print(f"[跳過] 不支援的副檔名 {ext}: {filename}")
            continue

        # 如果同一個年月已經有資料，印出提醒（保留後讀到的）
        if ym in df_dict:
            print(f"[注意] {ym} 重複，原檔被覆蓋 -> {filename}")

        df_dict[ym] = df
        print(f"[OK] {ym} <- {filename}  ({ext})  shape={df.shape}")

    except Exception as e:
        print(f"[錯誤] {filename}: {e}")

# 
print(f"\n共讀取 {len(df_dict)} 個月份")
print(f"年月範圍: {min(df_dict.keys())} ~ {max(df_dict.keys())}")

# ====== 檢查有沒有缺月 ======
# 設定預期應有年月 (201701~202512)
expected = []
for y in range(2017, 2026):
    for m in range(1, 13):
        expected.append(f"{y}{m:02d}")

# 找出 df_dict裡是否有缺少月份
missing = [ym for ym in expected if ym not in df_dict]
if missing:
    print(f"缺少的月份 ({len(missing)}): {missing}")
else:
    print("201701~202512 全部到齊！")

[OK] 201701 <- 201701月報表.htm  (.htm)  shape=(21, 13)
[OK] 201702 <- 201702月報表.htm  (.htm)  shape=(21, 13)
[OK] 201703 <- 201703月報表.htm  (.htm)  shape=(21, 13)
[OK] 201704 <- 201704月報表.htm  (.htm)  shape=(21, 13)
[OK] 201705 <- 201705月報表.htm  (.htm)  shape=(21, 13)
[OK] 201706 <- 201706月報表.htm  (.htm)  shape=(21, 13)
[OK] 201707 <- 201707月報表.htm  (.htm)  shape=(21, 13)
[OK] 201708 <- 201708月報表.htm  (.htm)  shape=(21, 13)
[OK] 201709 <- 201709月報表.htm  (.htm)  shape=(21, 13)
[OK] 201710 <- 201710月報表.htm  (.htm)  shape=(21, 13)
[OK] 201711 <- 201711月報表.htm  (.htm)  shape=(21, 13)
[OK] 201712 <- 201712觀光旅館營運月報表彙整表.xlsx  (.xlsx)  shape=(386, 19)
[OK] 201801 <- 201801月報表-修.htm  (.htm)  shape=(21, 13)
[OK] 201802 <- 201802月報表.htm  (.htm)  shape=(21, 13)
[OK] 201803 <- 201803月報表.htm  (.htm)  shape=(21, 13)
[OK] 201804 <- 201804月報表.htm  (.htm)  shape=(21, 13)
[OK] 201805 <- 201805月報表.html  (.html)  shape=(21, 13)
[OK] 201806 <- 201806月報表.htm  (.htm)  shape=(21, 13)
[OK] 201807 <- 201807月報表.html 

In [ ]:
def clean_df(df_dict):
    """
    每個 df 對應一個月份的工作表，格式來自觀光局旅館營運統計。(201701~202512)

    參數:
        df_dict: {年月字串: DataFrame} 的字典（原始讀入，尚未清洗）

    回傳:
        df_dict_clean: 清洗後的字典，欄位統一為：
        地區、類型、住用數、住用率、平均房價、客房收入、餐廳收入、總營收
    """
    df_dict_clean = {}
    for ym, df in df_dict.items():
        # NFKC 正規化 
        # 欄位名
        df.columns = [unicodedata.normalize('NFKC', str(col)) for col in df.columns]
        # 資料值
        df = df.apply(lambda col: col.map(lambda x: unicodedata.normalize('NFKC', x) if isinstance(x, str) else x))

        # === 第一步：定位表頭列（含「地區名稱」的那一行） ===
        # 如果欄位名稱本身就含「地區名稱」，且資料列裡沒有重複出現 → 表頭已正確，不用動
        if (any('地區名稱' in str(col) for col in df.columns)
            and not df.apply(lambda col: col.astype(str).str.contains('地區名稱', na=False)).any(axis=1).any()):
            header_idx = None  # 已經是 header 了，不需要再調
        else:
            # 否則在資料列中搜尋「地區名稱」，把那一行當作真正的表頭
            mask = df.apply(lambda col: col.astype(str).str.contains('地區名稱', na=False)).any(axis=1)
            if mask.sum() == 0:
                print(f"⚠️ {ym} 找不到 '地區名稱'，跳過")
                continue
            header_idx = df[mask].index[0]  # 取出包含「地區名稱」的第一個

        # === 第二步：定位結尾列（含「總計」的那一行），作為資料截斷點 ===
        mask = df.apply(lambda col: col.astype(str).str.contains('總計', na=False)).any(axis=1)
        if mask.sum() == 0:
            print(f"⚠️ {ym} 找不到 '總計'，跳過")
            continue
        end_idx = df[mask].index[0]

        # === 第三步：根據表頭位置切出有效資料範圍 ===
        if header_idx is not None:
            df.columns = df.iloc[header_idx]            # 用該行當新欄位名
            df = df.iloc[header_idx + 1:end_idx].reset_index(drop=True)   # header_idx的下一行 ~ end_idx的前一行
        else:
            df = df.iloc[:end_idx].reset_index(drop=True)  # 欄位名稱不須變動，只需要end_idx的前一行

        # === 第四步：移除小計列 只留國際和一般 ===
        df = df[~df.apply(lambda row: row.astype(str).str.contains('小計')).any(axis=1)]

        # === 第五步：找出含「國際」字樣的欄位，重新命名為「類型」 ===
        # 逐欄確認是否有包含國際，將該欄命名為 "類型"
        for i, col in enumerate(df.columns):
            if df.iloc[:, i].astype(str).str.contains('國際', na=False).any():  
                cols = list(df.columns)
                cols[i] = '類型'
                df.columns = cols
                break

        # === 第六步：清除欄位值中的英文字母（原始資料混有英文註記） ===
        df = df.replace(r'[A-Za-z]', '', regex=True)

        # === 第七步：只保留需要的欄位 ===
        keep_cols = ['地區', '類型', '住用數', '住用率', '平均房價', '收入', '總營收']
        # 先把 tuple 型欄位名（多層 header 合併後可能出現）統一轉成字串
        df.columns = ['_'.join(str(x) for x in col).strip('_') if isinstance(col, tuple) else str(col) for col in df.columns]

        # 建立正規表達式 pattern
        pattern = '|'.join(keep_cols)  # '地區|類型|住用數|住用率|平均房價|收入|總營收'

        # 用 pattern 標記每個欄位：符合 = True，不符合 = False
        mask = df.columns.str.contains(pattern, na=False)

        # 印出即將被移除的欄位（mask 為 False 的）
        print('被移除的欄位：')
        print(df.columns[~mask].tolist())

        # 只保留符合的欄位
        df = df.loc[:, mask]


        # === 第八步：統一欄位命名 ===
        # 用關鍵字匹配的方式，把各年度可能不同的原始欄位名對應到統一名稱
        col_mapping = {
            ('地區',):       '地區',
            ('類型',):       '類型',
            ('住用數',):     '住用數',
            ('住用率',):     '住用率',
            ('平均房價',):   '平均房價',
            ('房', '收入'):  '客房收入',    # 原始欄位可能叫「客房收入」或「房間收入」等
            ('餐', '收入'):  '餐廳收入',    # 原始欄位可能叫「餐廳收入」或「餐飲收入」等
            ('總營業',):     '總營業收入',
        }

        rename = {}
        for col in df.columns: # 確認原始欄位
            for keywords, new_name in col_mapping.items(): # 對應所有col_mapping
                if all(k in str(col) for k in keywords):  # 如果對應到keywords，轉換成new_name，並結束col_mapping for loop，檢查其他cols
                    rename[col] = new_name
                    break

        df = df.rename(columns=rename)

        # reindex 確保輸出欄位順序一致，缺少的欄位會自動填 NaN
        final_cols= col_mapping.values()  #  ['地區', '類型', '住用數', '住用率', '平均房價', '客房收入', '餐廳收入', '總營收']
        df = df.reindex(columns=final_cols)

        # 設定年月為 datetime
        df["年月"] = str(ym)
        df["年月"] = pd.to_datetime(df["年月"] ,format="%Y%m").dt.to_period("M")

        # 存入 df_dict_clean
        df_dict_clean[ym] = df
    return df_dict_clean

In [22]:
df_dict_clean= clean_df(df_dict)

被移除的欄位：
["('觀光旅館營運月報表單月-彙整表 (Monthly Report on Tourist Hotel Operations in Taiwan)', '2017年1月至2017年1月(Data for 2017/1~2017/1)', '各部門職工概況(No. of Employees)', '客房部 Room Dep.')", "('觀光旅館營運月報表單月-彙整表 (Monthly Report on Tourist Hotel Operations in Taiwan)', '2017年1月至2017年1月(Data for 2017/1~2017/1)', '各部門職工概況(No. of Employees)', '餐飲部 F&B Dep.')", "('觀光旅館營運月報表單月-彙整表 (Monthly Report on Tourist Hotel Operations in Taiwan)', '2017年1月至2017年1月(Data for 2017/1~2017/1)', '各部門職工概況(No. of Employees)', '管理部 Adm. Dep.')", "('觀光旅館營運月報表單月-彙整表 (Monthly Report on Tourist Hotel Operations in Taiwan)', '2017年1月至2017年1月(Data for 2017/1~2017/1)', '各部門職工概況(No. of Employees)', '其他部門 Other Dep.')", "('觀光旅館營運月報表單月-彙整表 (Monthly Report on Tourist Hotel Operations in Taiwan)', '2017年1月至2017年1月(Data for 2017/1~2017/1)', '各部門職工概況(No. of Employees)', '員工合計 Total')"]
被移除的欄位：
["('觀光旅館營運月報表單月-彙整表 (Monthly Report on Tourist Hotel Operations in Taiwan)', '2017年2月至2017年2月(Data for 2017/2~2017/2)', '各部門職工概況(No. of Employees)', '

In [23]:
for ym,df in df_dict_clean.items():
    if df.isnull().any().any(): 
        print(ym)
        print(df.to_string())

201901
       地區  類型     住用數     住用率  平均房價       客房收入        餐廳收入       總營業收入       年月
0    臺北地區  國際  188433  0.7102   NaN  862789389  1355267657  2514117150  2019-01
1    臺北地區  一般   57649  0.7015   NaN  223500186   244689802   540525006  2019-01
3    高雄地區  國際   68219  0.6039   NaN  161839658   266542073   470763669  2019-01
4    高雄地區  一般    5349  0.4346   NaN   14556936    19364394    36857594  2019-01
6    臺中地區  國際   21474  0.6092   NaN   53784311   107210605   186663883  2019-01
7    臺中地區  一般    7355  0.6798   NaN   21133244    36327715    63541052  2019-01
9    花蓮地區  國際   16481  0.4134   NaN   39483652    35525375    81968544  2019-01
11    風景區  國際   42417  0.6403   NaN  213677668   109371202   354671258  2019-01
12    風景區  一般    7149  0.2991   NaN   23469577    13093937    37516696  2019-01
14  桃竹苗地區  國際   49916  0.6992   NaN  144481707   217154994   395182121  2019-01
15  桃竹苗地區  一般   25424  0.5892   NaN   65407776    65673221   137929834  2019-01
17   其他地區  國際   56936  0.5294   N

#### 201901 平均房價缺失
#### 201906 整體資料向右偏移，後續再手動處理
#### 202003 客房收入缺失

### 除了201906以外，其餘皆為原資料缺失

#### 2021-07 的澎湖縣_一般 與 金門縣_一般的平均房價原始資料均為False 並非資料處理錯誤

In [24]:
# 合併資料，按照年月排序

df_all = pd.concat(df_dict_clean.values(), ignore_index=False)
df_all = df_all.sort_values('年月').reset_index(drop=True)
df_all

,地區,類型,住用數,住用率,平均房價,客房收入,餐廳收入,總營業收入,年月
0,臺北地區,國際,178343.0,64.74%,4790,854187080,1510697840,2665345399,2017-01
1,臺北地區,一般,51804.0,70.16%,3955,204868323,271519132,544203664,2017-01
2,高雄地區,國際,51358.0,57.35%,2817,144680577,306287221,514871710,2017-01
3,高雄地區,一般,2673.0,34.49%,2966,7928758,4861179,12864063,2017-01
4,臺中地區,國際,22051.0,62.56%,2855,62958665,130037260,223968272,2017-01
...,...,...,...,...,...,...,...,...,...
2562,新竹市,一般,0.0,0.0,0,0,0,0,2025-12
2563,嘉義市,國際,4404.0,0.5799,2468,10870188,9372731,21887893,2025-12
2564,嘉義市,一般,1774.0,0.4769,1884,3341944,5832152,9200861,2025-12
2565,金門縣,國際,0.0,0.0,0,0,0,0,2025-12


In [ ]:
# 產出資料
df_all.to_excel("觀光旅館_all.xlsx",index=False)

### 201906 異常資料處理
該月份的 PDF 來源導致欄位結構與其他月份不同，無法自動清洗，改為匯出後手動修正。

In [40]:
pdf_df= df_dict["201906"].iloc[2:,[0,1,7]]
pdf_df = pdf_df[~pdf_df.apply(lambda row: row.astype(str).str.contains('小計')).any(axis=1)]
pdf_df

,觀光旅館營運月報表單月-彙整表,Unnamed: 0,Unnamed: 6
2,地區名稱,客房住用數,客房部
3,臺北地區,國際,"2,080,431,967"
4,臺北地區,一般,"433,825,700"
6,高雄地區,國際,"403,344,796"
7,高雄地區,一般,"33,633,800"
...,...,...,...
101,NaN,住用及營收概況,NaN
102,旅館名稱,客房數,總營業收入
103,全國大飯店,178,"26,637,113"
104,長榮桂冠酒店(台中),354,"45,270,319"


In [42]:
# 201906的pdf資料與其他資料欄位不同，導致缺失值，因只有當月有這個狀況，因此直接載出EXCEL再手動更改
pdf_df.to_excel("201906.xlsx",index=False)